# Truthprint — 실증 데이터 핸드오프 (Kaggle · Colab · Local)

**셀을 위에서부터 순서대로 실행**하면 실측 실험용 데이터(실제 번역문 + 사람 주석 초안)를
만들어 **zip으로 내려받는 것**까지 끝냅니다. 이 노트북은 Kaggle·Colab·로컬 Jupyter를
**자동 감지**하므로 어디서 열어도 동작합니다.

> ⚠️ **Kaggle 사용자 필독:** 오른쪽 패널 **Settings → Internet → On** 을 켜세요
> (전화 인증된 계정 필요). 인터넷이 꺼져 있으면 저장소 clone·설치·번역이 실패합니다.
> 그리고 **Settings → Accelerator 는 None(CPU)** 로 두면 됩니다(GPU 불필요).

## STEP 0 — 환경 감지 + 저장소 클론 + 설치

In [ ]:
import os, sys, subprocess

# 작업 베이스 디렉터리 자동 감지 (Kaggle / Colab / 로컬)
if os.path.isdir('/kaggle/working'):
    BASE = '/kaggle/working'
elif os.path.isdir('/content'):
    BASE = '/content'
else:
    BASE = os.getcwd()
REPO = os.path.join(BASE, 'truthprint')
print('BASE =', BASE)
print('REPO =', REPO)

if not os.path.isdir(REPO):
    r = subprocess.run(['git','clone','--depth','1',
                        'https://github.com/leemgs/truthprint', REPO])
    if r.returncode != 0:
        raise RuntimeError('git clone 실패 -> 인터넷이 꺼져 있는지 확인하세요 '
                           '(Kaggle: Settings > Internet > On).')

# 편집가능(editable) 설치
r = subprocess.run([sys.executable,'-m','pip','install','-q','-e',
                    os.path.join(REPO,'code')+'[dev]'])
if r.returncode != 0:
    print('pip install 경고: 계속 진행합니다(테스트에는 sys.path로 충분).')
sys.path.insert(0, os.path.join(REPO,'code'))

# 이후 셀에서 쓸 경로
SRC  = os.path.join(REPO, 'handoff', 'samples')
WORK = os.path.join(BASE, 'my_handoff_data')
assert os.path.isdir(SRC), f'경로 없음: {SRC} (clone 실패로 보임)'
print('OK  SRC =', SRC)
print('OK  WORK will be =', WORK)

## STEP 1 — 코드가 실제로 도는지 확인 (권장)

In [ ]:
from truthprint.cli import main as tp_main
print('--- selftest ---');  tp_main(['selftest'])
print('--- challenge ---'); tp_main(['challenge'])

## STEP 2 — 실제-규모 소스 생성  ⭐ (tiny 샘플 대신)

`handoff/samples/` 의 예제는 **형식 확인용 tiny 데이터**(2문장, 2비트 태그)라 실측이
불가능합니다. 이 셀은 설치된 `truthprint` 패키지로 **실제-규모 소스**를 직접 생성합니다
(적절한 페이로드/태그, 부호율 0.5, ECC 여유 확보). 규모는 아래 변수로 조절하세요.

> **주의(정직한 기대치):** 현재 폐쇄도메인 carrier는 영어의 *태(voice)/시간구 위치*라는
> 표면 특징입니다. 직접 KO/HI 번역은 이를 대개 정규화(소거)하므로 그 조건은 소거가 많고,
> **round-trip(영어로 복귀)** 조건이 가장 잘 복구됩니다. 이는 버그가 아니라 논문이 말한
> "직접 번역 강건성은 광역 semantic frontend가 필요"라는 지점을 그대로 보여줍니다.

> 문서 수 x 문장 수가 클수록 신뢰구간은 좋아지지만 **STEP 5 주석 부담**도 커집니다.

In [ ]:
import os, json, random
from truthprint.core import Truthprint
from truthprint import challenge as ch

# ---- 실험 규모 (원하는 대로 조절) ----
N_DOCS        = 4      # 문서 수 (조건별 데이터 포인트 수)
SENTS_PER_DOC = 16     # 문서당 문장 수 (carrier = 2 x 문장)
MSG_LEN, TAG_BITS = 8, 16   # 페이로드 k = 24; 부호 [2*S, k]
GEN_SEED = 20270922
KEY = b'truthprint-challenge-key-01234567'[:32]  # eval_handoff 기본 키와 동일

os.makedirs(WORK, exist_ok=True)
rng = random.Random(GEN_SEED)
rows = []
for d in range(N_DOCS):
    doc_id = f'D{d+1:04d}'
    facts = [ch.sample_fact(rng) for _ in range(SENTS_PER_DOC)]
    core = Truthprint(KEY, msg_len=MSG_LEN, tag_bits=TAG_BITS, code_n=2*SENTS_PER_DOC)
    msg = [rng.randrange(2) for _ in range(MSG_LEN)]
    nonce = bytes(rng.randrange(256) for _ in range(12))
    opts = core.encode(ch.doc_invariants(facts), msg, nonce)
    wm = {}; frows = []
    for i, f in enumerate(facts):
        sid = f'{doc_id}-s{i+1}'
        wm[sid] = ch.realize(f, opts[2*i], opts[2*i+1])
        frows.append(dict(ch.ext_invariants(f), sent_id=sid))
    split = 'test' if d < max(1, N_DOCS//2) else 'calibration'
    rows.append({'doc_id':doc_id,'split':split,'lang_src':'en','domain':'news',
        'scheme':'truthprint','key_id':'k1','nonce_hex':nonce.hex(),
        'message_bits':''.join(map(str,msg)),
        'code':{'n':2*SENTS_PER_DOC,'k':MSG_LEN+TAG_BITS,'msg_len':MSG_LEN,'tag_bits':TAG_BITS},
        'facts':frows,'watermarked_text':wm})

with open(os.path.join(WORK,'01_source_items.jsonl'),'w',encoding='utf-8') as fh:
    for r in rows: fh.write(json.dumps(r, ensure_ascii=False)+'\n')
open(os.path.join(WORK,'02_transformations.jsonl'),'w',encoding='utf-8').close()
open(os.path.join(WORK,'03_annotations.jsonl'),'w',encoding='utf-8').close()
with open(os.path.join(WORK,'04_human_factuality.csv'),'w',encoding='utf-8') as fh:
    fh.write('pair_id,doc_id,sent_id,transform_id,kind,field_if_altering,human_equivalent,annotator_id,notes\n')
open(os.path.join(WORK,'05_baseline_outputs.jsonl'),'w',encoding='utf-8').close()
test=[r['doc_id'] for r in rows if r['split']=='test']
cal =[r['doc_id'] for r in rows if r['split']=='calibration']
json.dump({'calibration':cal,'test':test}, open(os.path.join(WORK,'split.json'),'w'))
rate = (MSG_LEN+TAG_BITS)/(2*SENTS_PER_DOC)
print(f'generated {N_DOCS} docs x {SENTS_PER_DOC} sents; code rate {rate:.2f}; '
      f'tolerates ~{int((1-rate)*2*SENTS_PER_DOC)} erasures/doc')
print('working folder:', WORK)

## STEP 3 — 번역할 원문 문장 목록 뽑기

In [ ]:
import json, csv
rows = []
with open(os.path.join(WORK,'01_source_items.jsonl'), encoding='utf-8') as fh:
    for line in fh:
        r = json.loads(line)
        for sid, text in r['watermarked_text'].items():
            rows.append((r['doc_id'], sid, text))
with open(os.path.join(WORK,'to_translate.csv'),'w',newline='',encoding='utf-8') as fh:
    w = csv.writer(fh); w.writerow(['doc_id','sent_id','source_text']); w.writerows(rows)
print(f'{len(rows)} sentences -> to_translate.csv')
for _, sid, t in rows:
    print(' ', sid, '|', t)

## STEP 4 — 실제 번역 만들기  ⭐ (웹 API: MyMemory 우선, Google 폴백)

EN→KO, EN→HI, round-trip(EN→KO→EN)을 실제 수행해 `02_transformations.jsonl` 을 채웁니다.

> **왜 MyMemory 우선?** Kaggle/Colab의 데이터센터 IP는 Google 무료 엔드포인트에서 자주
> `TooManyRequests(429)` 로 차단됩니다. **MyMemory**(무료, 키 불필요)는 클라우드 IP에서도
> 대체로 동작합니다. 이 셀은 MyMemory→Google 순서로 시도하고, 문장 단위로 실패는 건너뜁니다.
>
> - 익명 한도가 부족하면 `EMAIL` 에 본인 이메일을 넣으세요(MyMemory 일일 한도 상향, 무료).
> - **웹 API가 계속 막히면 아래 STEP 4B(로컬 NLLB)** 를 대신 실행하세요 — 인터넷 차단 IP에서도
>   모델을 내려받아 **오프라인으로** 번역합니다(가장 확실).

In [ ]:
import subprocess, sys, json, os, time
subprocess.run([sys.executable,'-m','pip','install','-q','deep-translator'])
from deep_translator import GoogleTranslator, MyMemoryTranslator

EMAIL = ''          # (선택) MyMemory 한도 상향용 이메일. 비워도 됨.
THROTTLE = 1.0      # 요청 간 지연(초)
_REGION = {'en':'en-US','ko':'ko-KR','hi':'hi-IN'}

def _mymemory(text, src, tgt):
    kw = {'email': EMAIL} if EMAIL else {}
    return MyMemoryTranslator(source=_REGION[src], target=_REGION[tgt], **kw).translate(text)
def _google(text, src, tgt):
    return GoogleTranslator(source=src, target=tgt).translate(text)
BACKENDS = [('MyMemory', _mymemory), ('Google', _google)]

def tr(text, src, tgt, tries=4):
    last = None
    for name, fn in BACKENDS:
        for i in range(tries):
            try:
                time.sleep(THROTTLE)
                r = fn(text, src, tgt)
                if r:
                    tr.system = f'deep-translator/{name}'
                    return r
            except Exception as e:
                last = e; wait = min(20, 2 ** i)
                print(f'    [{name}] 재시도 {i+1}/{tries} ({type(e).__name__}) -> {wait}s')
                time.sleep(wait)
        print(f'    [{name}] 실패 -> 다음 백엔드로')
    raise RuntimeError(f'모든 백엔드 실패: {last}. STEP 4B(로컬 NLLB)를 사용하세요.')
tr.system = 'deep-translator'

out, failed = [], []
for doc_id, sid, text in rows:
    try:
        ko = tr(text,'en','ko'); sysname = tr.system
        hi = tr(text,'en','hi'); rt = tr(ko,'ko','en')
    except Exception as e:
        print(f'  [skip] {sid}: {e}'); failed.append(sid); continue
    out += [
      {'transform_id':f'{sid}-ko','doc_id':doc_id,'sent_id':sid,'transform_type':'translation','direction':'en->ko','system':sysname,'params':{},'output_text':ko,'round_trip':False},
      {'transform_id':f'{sid}-hi','doc_id':doc_id,'sent_id':sid,'transform_type':'translation','direction':'en->hi','system':sysname,'params':{},'output_text':hi,'round_trip':False},
      {'transform_id':f'{sid}-rt','doc_id':doc_id,'sent_id':sid,'transform_type':'roundtrip_translation','direction':'en->ko->en','system':sysname,'params':{},'output_text':rt,'round_trip':True},
    ]
    print(f'  ok {sid} ({sysname})')

if out:
    with open(os.path.join(WORK,'02_transformations.jsonl'),'w',encoding='utf-8') as fh:
        for r in out:
            fh.write(json.dumps(r, ensure_ascii=False)+'\n')
print(f'\nwrote {len(out)} transformations for {len(rows)-len(failed)}/{len(rows)} sentences')
if failed:
    print('실패:', failed, '-> 잠시 후 이 셀 재실행, 또는 STEP 4B(로컬 NLLB) 사용.')
for r in out[:6]:
    print(' ', r['transform_id'], '|', r['output_text'])

## STEP 4B — (대안) 로컬 NLLB 번역  🛟 웹 API가 막힐 때만 실행

웹 번역 API가 계속 차단되면, Meta의 **NLLB-200**(연구용 MT, 오픈웨이트)을 내려받아
**로컬에서** 번역합니다. 인터넷은 모델 다운로드(최초 1회, 약 2.4GB)에만 필요하고 이후
CPU로 동작합니다(문장 수가 적으면 충분히 빠름). STEP 4 대신 이 셀을 실행하면 같은
`02_transformations.jsonl` 을 만듭니다.

> STEP 4가 이미 성공했다면 이 셀은 건너뛰세요.

In [ ]:
import subprocess, sys, json, os
subprocess.run([sys.executable,'-m','pip','install','-q','transformers','sentencepiece','torch'])
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

MODEL = 'facebook/nllb-200-distilled-600M'
SYSTEM = f'HuggingFace/{MODEL}'
NLLB = {'en':'eng_Latn','ko':'kor_Hang','hi':'hin_Deva'}
print('모델 로딩(최초 다운로드 ~2.4GB)...')
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL)

def tr(text, src, tgt):
    tok.src_lang = NLLB[src]
    enc = tok(text, return_tensors='pt')
    bos = tok.convert_tokens_to_ids(NLLB[tgt])
    gen = model.generate(**enc, forced_bos_token_id=bos, max_length=256)
    return tok.batch_decode(gen, skip_special_tokens=True)[0]

out = []
for doc_id, sid, text in rows:
    ko = tr(text,'en','ko'); hi = tr(text,'en','hi'); rt = tr(ko,'ko','en')
    out += [
      {'transform_id':f'{sid}-ko','doc_id':doc_id,'sent_id':sid,'transform_type':'translation','direction':'en->ko','system':SYSTEM,'params':{},'output_text':ko,'round_trip':False},
      {'transform_id':f'{sid}-hi','doc_id':doc_id,'sent_id':sid,'transform_type':'translation','direction':'en->hi','system':SYSTEM,'params':{},'output_text':hi,'round_trip':False},
      {'transform_id':f'{sid}-rt','doc_id':doc_id,'sent_id':sid,'transform_type':'roundtrip_translation','direction':'en->ko->en','system':SYSTEM,'params':{},'output_text':rt,'round_trip':True},
    ]
    print(f'  ok {sid}')
with open(os.path.join(WORK,'02_transformations.jsonl'),'w',encoding='utf-8') as fh:
    for r in out:
        fh.write(json.dumps(r, ensure_ascii=False)+'\n')
print(f'wrote {len(out)} transformations (NLLB)')
for r in out[:6]:
    print(' ', r['transform_id'], '|', r['output_text'])

## STEP 5 — 사람 주석 (초안 자동 생성 → 당신이 검토·수정)  ⭐

아래 셀은 **초안** `03_annotations.jsonl` 을 만듭니다(번역이 의미를 보존했다고 가정하고
원문 불변량을 채움). **번역문을 읽고 반드시 검토**하세요:

1. 의미(극성·수량·시간방향·양태·귀속·인과)가 바뀌었으면 해당 필드 수정 + `invariant_preserved=false`.
2. 태(voice)/시간구 위치 carrier가 사라졌으면 그 carrier `reliable=false`(→ 검출에서 erasure).

필드 정의: 리포의 `handoff/schemas/SCHEMA_KO.md`.

In [ ]:
import json, os
from truthprint import challenge as ch
# 영어 원문을 파싱해 '실제 커밋된' carrier(태/시간위치)를 시드로 사용
src_inv, src_car = {}, {}
with open(os.path.join(WORK,'01_source_items.jsonl'), encoding='utf-8') as fh:
    for line in fh:
        r = json.loads(line)
        for fct in r['facts']:
            src_inv[fct['sent_id']] = {k: fct[k] for k in ['agent','patient','predicate',
                'quantity','polarity','time_dir','modality','attribution','causation']}
        for sid, text in r['watermarked_text'].items():
            f, car = ch.parse(text)   # 폐쇄도메인 파서로 커밋된 carrier 복원
            src_car[sid] = ('active' if car[0][0]==0 else 'passive',
                            'front'  if car[1][0]==0 else 'end')

draft = []
with open(os.path.join(WORK,'02_transformations.jsonl'), encoding='utf-8') as fh:
    for line in fh:
        t = json.loads(line); sid = t['sent_id']
        v, tp = src_car[sid]
        draft.append({'transform_id':t['transform_id'],'annotator_id':'A1_DRAFT',
            'invariants_observed':dict(src_inv[sid]),
            'carriers_observed':[{'carrier':'voice','value':v,'reliable':True},
                                 {'carrier':'time_position','value':tp,'reliable':True}],
            'invariant_preserved':True,
            'notes':'AUTO-DRAFT(영어 원문 기준). 검토: (1)번역이 의미를 바꿨으면 해당 필드 수정+invariant_preserved=false; (2)번역이 이 태/시간위치를 유지하지 않으면 그 carrier reliable=false(=소거).'})
with open(os.path.join(WORK,'03_annotations.jsonl'),'w',encoding='utf-8') as fh:
    for r in draft:
        fh.write(json.dumps(r, ensure_ascii=False)+'\n')
print(f'wrote {len(draft)} DRAFT annotations (carrier=영어 원문 기준 시드) -> 검토 필요')
print('핵심: 직접 KO/HI 번역은 태/시간위치를 대개 유지하지 않으므로, 유지 안 된 carrier는 reliable=false 로 바꾸세요.')

## STEP 5B — 직접 번역 carrier 일괄 소거  🖊 (파일을 손으로 편집할 필요 없음)

Kaggle/Colab 파일 패널은 출력 파일을 브라우저에서 편집할 수 없습니다. 그래서 손으로
JSON을 고치는 대신 이 셀로 처리합니다: 직접 번역(`-ko`/`-hi`)은 영어의 태/시간위치
carrier를 대개 보존하지 않으므로 **일괄 `reliable=false`(=소거)** 로, round-trip(`-rt`)은
영어로 복귀하므로 시드값(`reliable=true`)을 **유지**합니다.

> 이는 '직접 번역은 표면 carrier 미보존'이라는 합리적 기본 가정입니다. 특정 문장이
> 예외(번역이 태/시간위치를 그대로 유지)라면 그 줄만 아래에서 True로 되돌리세요.

In [ ]:
import json, os
path = os.path.join(WORK, '03_annotations.jsonl')
rows = [json.loads(l) for l in open(path, encoding='utf-8')]
DIRECT = ('-ko', '-hi')   # 직접 번역 조건
changed = 0
for r in rows:
    if r['transform_id'].endswith(DIRECT):
        for c in r['carriers_observed']:
            if c.get('reliable', True):
                c['reliable'] = False; changed += 1
    # -rt(round-trip)는 유지
with open(path, 'w', encoding='utf-8') as fh:
    for r in rows:
        fh.write(json.dumps(r, ensure_ascii=False) + '\n')
print(f'직접번역 carrier {changed}개 -> reliable=false. -rt 유지. 이제 STEP 7 실행.')

## STEP 6 — (선택) 사람 의미동일성 / baseline
`04_human_factuality.csv`, `05_baseline_outputs.jsonl` 은 예제 형식대로 채우면 됩니다.
어려우면 건너뛰어도 검증은 통과합니다.

## STEP 7 — 검증기로 형식 점검 (READY 뜰 때까지)

In [ ]:
import subprocess, sys, os
subprocess.run([sys.executable, os.path.join(REPO,'handoff','validate_handoff.py'), WORK])

## STEP 8 — 결과 zip 만들기 → 나에게 전달

아래 셀이 `my_handoff_data.zip` 을 만듭니다.
- **Kaggle:** 오른쪽 **Output** 패널(또는 `/kaggle/working`)에서 zip을 다운로드하세요.
  (Save Version 후 Output 탭에서도 받을 수 있습니다.)
- **Colab:** 자동 다운로드가 뜹니다.

그 파일을 저에게 주시거나, 리포 브랜치에 올린 뒤 이 세션에 알려주세요:
```
my_handoff_data 채웠고 validate READY 떴어. 실측 실험 돌려서 논문 표 채워줘.
```

In [ ]:
import shutil, os
for _f in ['02_transformations.jsonl','03_annotations.jsonl']:
    _sz = os.path.getsize(os.path.join(WORK,_f))
    print(('OK ' if _sz>0 else '!! 비어있음 '), _f, _sz, 'bytes')
    if _sz==0:
        print('   -> STEP 4/4B(번역)와 STEP 5(주석)를 먼저 완료하세요. 지금 zip은 아직 NOT READY.')

zip_base = os.path.join(BASE,'my_handoff_data')
zip_path = shutil.make_archive(zip_base, 'zip', WORK)
print('created:', zip_path)

downloaded = False
try:
    from google.colab import files  # Colab 전용
    files.download(zip_path); downloaded = True
except Exception:
    pass
if not downloaded:
    rel = os.path.relpath(zip_path, os.getcwd())
    try:
        from IPython.display import FileLink, display
        print('아래 링크를 클릭해 다운로드하세요 (Kaggle/로컬):')
        display(FileLink(rel))
    except Exception:
        print('수동 다운로드:', zip_path)
    print('또는 Kaggle 오른쪽 파일 패널에서 /kaggle/working/my_handoff_data.zip 을 받으세요 (우클릭 -> Download).')